# Sequence-to-Sequence Machine Translation: Architecture Comparison

**Task:** English → French neural machine translation  
**Dataset:** Tatoeba (fra-eng), filtered to 21,228 sentence pairs  
**Architectures evaluated:** GRU (baseline), LSTM, Bi-LSTM, GRU + Attention, Transformer Encoder  
**Evaluation metrics:** ROUGE-1 F1, ROUGE-2 F1  

> **Best result:** GRU + Attention — ROUGE-1: 0.6309 | ROUGE-2: 0.4360


## 1. Setup & Imports

In [ ]:
!pip install -q torchmetrics

import torch
import torch.nn as nn
from torch import optim
import torch.nn.functional as F
import unicodedata
import string
import re
import random
import time
import math
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from torchmetrics.text.rouge import ROUGEScore
from tqdm import tqdm
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")


## 2. Constants

In [ ]:
SOS_token = 0
EOS_token = 1
MAX_LENGTH = 15


## 3. Data Download & Preparation

In [ ]:
!wget -q http://www.manythings.org/anki/fra-eng.zip
!unzip -o -q fra-eng.zip
!mkdir -p data
!mv fra.txt data/eng-fra.txt
print("Data downloaded and moved to data/eng-fra.txt")


### 3.1 Language class & text normalisation

- Unicode → ASCII normalisation  
- Lowercasing and punctuation handling  
- Vocabulary built word-by-word


In [ ]:
class Lang:
    """Vocabulary and word-index mappings for a single language."""
    def __init__(self, name):
        self.name = name
        self.word2index = {}
        self.word2count = {}
        self.index2word = {0: "SOS", 1: "EOS"}
        self.n_words = 2

    def addSentence(self, sentence):
        for word in sentence.split(' '):
            self.addWord(word)

    def addWord(self, word):
        if word not in self.word2index:
            self.word2index[word] = self.n_words
            self.word2count[word] = 1
            self.index2word[self.n_words] = word
            self.n_words += 1
        else:
            self.word2count[word] += 1


def unicodeToAscii(s):
    """Convert Unicode string to plain ASCII."""
    return ''.join(
        c for c in unicodedata.normalize('NFD', s)
        if unicodedata.category(c) != 'Mn'
    )

def normalizeString(s):
    """Lowercase, trim, and remove non-letter characters."""
    s = unicodeToAscii(s.lower().strip())
    s = re.sub(r'([.!?])', r' \1', s)
    s = re.sub(r'[^a-zA-Z.!?]+', r' ', s)
    return s

def readLangs(lang1, lang2, reverse=False):
    print("Reading lines...")
    lines = open(f'data/{lang1}-{lang2}.txt', encoding='utf-8').read().strip().split('\n')
    pairs = [[normalizeString(s) for s in l.split('\t')] for l in lines]
    if reverse:
        pairs = [[p[1], p[0]] for p in pairs]
        input_lang = Lang(lang2)
        output_lang = Lang(lang1)
    else:
        input_lang = Lang(lang1)
        output_lang = Lang(lang2)
    return input_lang, output_lang, pairs

def filterPair(p):
    """Keep only short sentences starting with common English prefixes."""
    eng_prefixes = (
        "i am ", "i m ",
        "he is", "he s ",
        "she is", "she s ",
        "you are", "you re ",
        "we are", "we re ",
        "they are", "they re "
    )
    return (len(p[0].split(' ')) < MAX_LENGTH
            and len(p[1].split(' ')) < MAX_LENGTH
            and p[1].startswith(eng_prefixes))

def filterPairs(pairs):
    return [pair for pair in pairs if filterPair(pair)]

def prepareData(lang1, lang2, reverse=False):
    input_lang, output_lang, pairs = readLangs(lang1, lang2, reverse)
    print(f"Read {len(pairs)} sentence pairs")
    pairs = filterPairs(pairs)
    print(f"Trimmed to {len(pairs)} sentence pairs")
    print("Counting words...")
    for pair in pairs:
        input_lang.addSentence(pair[0])
        output_lang.addSentence(pair[1])
    print(f"Vocabulary — {input_lang.name}: {input_lang.n_words} words, "
          f"{output_lang.name}: {output_lang.n_words} words")
    return input_lang, output_lang, pairs

# Prepare data (French → English direction, so French is input_lang)
input_lang, output_lang, pairs = prepareData('eng', 'fra', True)
print(random.choice(pairs))


### 3.2 Train / Test split (90 / 10)

In [ ]:
X = [i[0] for i in pairs]
y = [i[1] for i in pairs]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=42)
train_pairs = list(zip(X_train, y_train))
test_pairs  = list(zip(X_test,  y_test))
print(f"Train pairs: {len(train_pairs)}, Test pairs: {len(test_pairs)}")


### 3.3 Tensor conversion utilities

In [ ]:
def indexesFromSentence(lang, sentence):
    return [lang.word2index[word] for word in sentence.split(' ')]

def tensorFromSentence(lang, sentence):
    indexes = indexesFromSentence(lang, sentence)
    indexes.append(EOS_token)
    return torch.tensor(indexes, dtype=torch.long, device=device).view(-1, 1)

def tensorsFromPair(pair):
    input_tensor  = tensorFromSentence(input_lang,  pair[0])
    target_tensor = tensorFromSentence(output_lang, pair[1])
    return input_tensor, target_tensor


## 4. Model Architectures

Five encoder architectures are implemented, all paired with a GRU-based decoder  
(except LSTM variants which use an LSTM decoder to match state types).

| Exp | Encoder | Decoder | Key Feature |
|-----|---------|---------|-------------|
| 1 | GRU | GRU | Baseline |
| 2 | LSTM | LSTM | Internal cell state |
| 3 | Bi-LSTM | LSTM | Bidirectional context |
| 4 | GRU | GRU + Attention | Multiplicative attention |
| 5 | Transformer | GRU | Self-attention + mean pooling |


### 4.1 Encoder: GRU (Baseline)

In [ ]:
class EncoderRNN(nn.Module):
    """Single-direction GRU encoder."""
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(input_size, hidden_size)
        self.gru = nn.GRU(hidden_size, hidden_size)

    def forward(self, input, hidden):
        embedded = self.embedding(input).view(1, 1, -1)
        output, hidden = self.gru(embedded, hidden)
        return output, hidden

    def initHidden(self):
        return torch.zeros(1, 1, self.hidden_size, device=device)


### 4.2 Encoder: LSTM

In [ ]:
class EncoderLSTM(nn.Module):
    """LSTM encoder with separate hidden and cell states."""
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(input_size, hidden_size)
        self.lstm = nn.LSTM(hidden_size, hidden_size)

    def forward(self, input, hidden):
        embedded = self.embedding(input).view(1, 1, -1)
        output, hidden = self.lstm(embedded, hidden)
        return output, hidden

    def initHidden(self):
        h = torch.zeros(1, 1, self.hidden_size, device=device)
        c = torch.zeros(1, 1, self.hidden_size, device=device)
        return (h, c)


### 4.3 Encoder: Bidirectional LSTM

In [ ]:
class EncoderBiLSTM(nn.Module):
    """Bidirectional LSTM encoder; hidden states merged via summation before passing to decoder."""
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(input_size, hidden_size)
        self.lstm = nn.LSTM(hidden_size, hidden_size, bidirectional=True, batch_first=False)

    def forward(self, input, hidden):
        embedded = self.embedding(input).view(1, 1, -1)
        output, hidden = self.lstm(embedded, hidden)
        return output, hidden

    def initHidden(self):
        h = torch.zeros(2, 1, self.hidden_size, device=device)
        c = torch.zeros(2, 1, self.hidden_size, device=device)
        return (h, c)


### 4.4 Encoder: Transformer

In [ ]:
class TransformerEncoder(nn.Module):
    """
    PyTorch TransformerEncoder with learnable positional embeddings.
    Returns encoder_outputs (seq_len, 1, hidden_size) and a mean-pooled
    context vector (1, 1, hidden_size) used as the decoder initial state.
    """
    def __init__(self, input_size, hidden_size, num_layers=2, nhead=8, max_len=100):
        super().__init__()
        self.hidden_size = hidden_size
        self.embedding    = nn.Embedding(input_size, hidden_size)
        self.pos_embedding = nn.Embedding(max_len, hidden_size)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_size, nhead=nhead,
            dim_feedforward=4 * hidden_size, dropout=0.1, batch_first=False
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

    def forward(self, input_tensor):
        seq_len   = input_tensor.size(0)
        emb       = self.embedding(input_tensor)
        positions = torch.arange(seq_len, device=device).unsqueeze(1)
        pos_emb   = self.pos_embedding(positions)
        x         = emb + pos_emb
        enc_out   = self.transformer(x)               # (seq_len, 1, hidden_size)
        context   = enc_out.mean(dim=0, keepdim=True) # (1, 1, hidden_size)
        return enc_out, context

    def initHidden(self):
        return torch.zeros(1, 1, self.hidden_size, device=device)


### 4.5 Decoders

In [ ]:
class Decoder(nn.Module):
    """Standard GRU decoder used with GRU / Transformer encoders."""
    def __init__(self, hidden_size, output_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(output_size, hidden_size)
        self.gru = nn.GRU(hidden_size, hidden_size)
        self.out = nn.Linear(hidden_size, output_size)
        self.softmax = nn.LogSoftmax(dim=1)

    def forward(self, input, hidden):
        embedded = self.embedding(input).view(1, 1, -1)
        output   = F.relu(embedded)
        output, hidden = self.gru(output, hidden)
        output   = self.softmax(self.out(output[0]))
        return output, hidden

    def initHidden(self):
        return torch.zeros(1, 1, self.hidden_size, device=device)


class DecoderLSTM(nn.Module):
    """LSTM decoder used with LSTM / Bi-LSTM encoders."""
    def __init__(self, hidden_size, output_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(output_size, hidden_size)
        self.lstm = nn.LSTM(hidden_size, hidden_size)
        self.out  = nn.Linear(hidden_size, output_size)
        self.softmax = nn.LogSoftmax(dim=1)

    def forward(self, input, hidden):
        embedded = self.embedding(input).view(1, 1, -1)
        output   = F.relu(embedded)
        output, hidden = self.lstm(output, hidden)
        output   = self.softmax(self.out(output[0]))
        return output, hidden

    def initHidden(self):
        h = torch.zeros(1, 1, self.hidden_size, device=device)
        c = torch.zeros(1, 1, self.hidden_size, device=device)
        return (h, c)


### 4.6 Attention module & attention-augmented decoder

In [ ]:
class AttentionModule(nn.Module):
    """
    Vectorised multiplicative (Bahdanau-style) attention.
    Computes a weighted context vector over all encoder outputs.
    """
    def __init__(self, hidden_size):
        super().__init__()
        self.attn = nn.Linear(hidden_size * 2, hidden_size)
        self.v    = nn.Linear(hidden_size, 1, bias=False)

    def forward(self, decoder_hidden, encoder_outputs):
        """
        decoder_hidden  : (1, 1, hidden_size)
        encoder_outputs : (max_length, hidden_size)
        Returns attention weights (1, 1, max_length) and context (1, 1, hidden_size).
        """
        max_length     = encoder_outputs.size(0)
        hidden_repeated = decoder_hidden.squeeze(0).repeat(max_length, 1)
        energy         = torch.tanh(self.attn(torch.cat([hidden_repeated, encoder_outputs], dim=1)))
        attention      = self.v(energy).squeeze(1)
        attn_weights   = F.softmax(attention, dim=0).unsqueeze(0).unsqueeze(0)
        context        = torch.bmm(attn_weights, encoder_outputs.unsqueeze(0))
        return attn_weights, context


class DecoderWithAttention(nn.Module):
    """GRU decoder augmented with multiplicative attention over encoder outputs."""
    def __init__(self, hidden_size, output_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.embedding   = nn.Embedding(output_size, hidden_size)
        self.gru         = nn.GRU(hidden_size, hidden_size)
        self.out         = nn.Linear(hidden_size, output_size)
        self.softmax     = nn.LogSoftmax(dim=1)
        self.attention   = AttentionModule(hidden_size)

    def forward(self, input, hidden, encoder_outputs):
        embedded = self.embedding(input).view(1, 1, -1)
        output   = F.relu(embedded)
        output, hidden = self.gru(output, hidden)
        _, context = self.attention(hidden, encoder_outputs)
        output = self.softmax(self.out(context[0]))
        return output, hidden

    def initHidden(self):
        return torch.zeros(1, 1, self.hidden_size, device=device)


## 5. Training & Evaluation Utilities

**Shared hyperparameters:**
- Hidden dimension: 256  
- Epochs: 3  
- Teacher-forcing ratio: 0.5  
- Optimiser: SGD, lr = 0.01  
- Loss: NLLLoss


In [ ]:
teacher_forcing_ratio = 0.5


# ── Timing helpers ────────────────────────────────────────────────────────
def asMinutes(s):
    m = math.floor(s / 60); s -= m * 60
    return f'{int(m)}m {int(s)}s'

def timeSince(since, percent):
    now = time.time(); s = now - since
    rs  = s / percent - s
    return f'{asMinutes(s)} (- {asMinutes(rs)})'


# ── Generic train step (RNN / Transformer) ───────────────────────────────
def train(input_tensor, target_tensor,
          encoder, decoder,
          encoder_optimizer, decoder_optimizer,
          criterion, max_length=MAX_LENGTH,
          encoder_type="rnn"):
    encoder_hidden = encoder.initHidden()
    encoder_optimizer.zero_grad()
    decoder_optimizer.zero_grad()

    input_length  = input_tensor.size(0)
    target_length = target_tensor.size(0)

    # ── Encoding ──────────────────────────────────────────────────────────
    if encoder_type == "transformer":
        _, context    = encoder(input_tensor)
        decoder_hidden = context
    else:
        test_output, _    = encoder(input_tensor[0], encoder.initHidden())
        enc_out_size      = test_output.size(2)
        encoder_outputs   = torch.zeros(max_length, enc_out_size, device=device)
        encoder_hidden    = encoder.initHidden()

        for ei in range(input_length):
            encoder_output, encoder_hidden = encoder(input_tensor[ei], encoder_hidden)
            encoder_outputs[ei] = encoder_output[0, 0]

        if isinstance(encoder_hidden, tuple):          # LSTM / Bi-LSTM
            h, c = encoder_hidden
            if h.size(0) == 2:                         # Bidirectional → merge
                h = h.sum(dim=0, keepdim=True)
                c = c.sum(dim=0, keepdim=True)
            encoder_hidden = (h, c)

        decoder_hidden = encoder_hidden

    # ── Decoding ──────────────────────────────────────────────────────────
    decoder_input        = torch.tensor([[SOS_token]], device=device)
    use_teacher_forcing  = random.random() < teacher_forcing_ratio
    loss = 0

    if use_teacher_forcing:
        for di in range(target_length):
            decoder_output, decoder_hidden = decoder(decoder_input, decoder_hidden)
            loss += criterion(decoder_output, target_tensor[di])
            decoder_input = target_tensor[di]
    else:
        for di in range(target_length):
            decoder_output, decoder_hidden = decoder(decoder_input, decoder_hidden)
            topv, topi = decoder_output.topk(1)
            decoder_input = topi.squeeze().detach()
            if decoder_input.dim() == 0: decoder_input = decoder_input.view(1, 1)
            elif decoder_input.dim() == 1: decoder_input = decoder_input.view(1, 1)
            loss += criterion(decoder_output, target_tensor[di])
            if decoder_input.item() == EOS_token:
                break

    loss.backward()
    encoder_optimizer.step()
    decoder_optimizer.step()
    return loss.item() / target_length


def trainIters(encoder, decoder, epochs,
               print_every=1000, learning_rate=0.01, encoder_type="rnn"):
    """Train for multiple epochs; returns (loss_history, iteration_list)."""
    start              = time.time()
    print_loss_total   = 0
    loss_history       = []
    iterations         = []

    encoder_optimizer = optim.SGD(encoder.parameters(), lr=learning_rate)
    decoder_optimizer = optim.SGD(decoder.parameters(), lr=learning_rate)
    criterion         = nn.NLLLoss()

    n_iters = len(train_pairs) * epochs
    it = 0

    for epoch in range(epochs):
        print(f"Epoch {epoch + 1}/{epochs}")
        for training_pair in train_pairs:
            input_tensor, target_tensor = tensorsFromPair(training_pair)
            loss = train(input_tensor, target_tensor,
                         encoder, decoder,
                         encoder_optimizer, decoder_optimizer,
                         criterion, encoder_type=encoder_type)
            print_loss_total += loss
            it += 1
            if it % print_every == 0:
                avg = print_loss_total / print_every
                loss_history.append(avg)
                iterations.append(it)
                print_loss_total = 0
                print(f"{timeSince(start, it / n_iters)} ({it}/{n_iters}) loss={avg:.4f}")

    print("Training complete!")
    return loss_history, iterations


# ── Attention-specific train step ────────────────────────────────────────
def train_with_attention(input_tensor, target_tensor,
                         encoder, decoder,
                         encoder_optimizer, decoder_optimizer,
                         criterion, max_length=MAX_LENGTH):
    encoder_hidden    = encoder.initHidden()
    encoder_optimizer.zero_grad()
    decoder_optimizer.zero_grad()

    input_length  = input_tensor.size(0)
    target_length = target_tensor.size(0)
    encoder_outputs = torch.zeros(max_length, encoder.hidden_size, device=device)

    for ei in range(input_length):
        encoder_output, encoder_hidden = encoder(input_tensor[ei], encoder_hidden)
        encoder_outputs[ei] = encoder_output[0, 0]

    decoder_hidden = encoder_hidden
    decoder_input  = torch.tensor([[SOS_token]], device=device)
    use_teacher_forcing = random.random() < teacher_forcing_ratio
    loss = 0

    if use_teacher_forcing:
        for di in range(target_length):
            decoder_output, decoder_hidden = decoder(decoder_input, decoder_hidden, encoder_outputs)
            loss += criterion(decoder_output, target_tensor[di])
            decoder_input = target_tensor[di]
    else:
        for di in range(target_length):
            decoder_output, decoder_hidden = decoder(decoder_input, decoder_hidden, encoder_outputs)
            topv, topi = decoder_output.topk(1)
            decoder_input = topi.squeeze().detach()
            if decoder_input.dim() == 0: decoder_input = decoder_input.view(1, 1)
            elif decoder_input.dim() == 1: decoder_input = decoder_input.view(1, 1)
            loss += criterion(decoder_output, target_tensor[di])
            if decoder_input.item() == EOS_token:
                break

    loss.backward()
    encoder_optimizer.step()
    decoder_optimizer.step()
    return loss.item() / target_length


def trainIters_attention(encoder, decoder, epochs,
                         print_every=1000, learning_rate=0.01):
    """Train attention model; returns (loss_history, iteration_list)."""
    start            = time.time()
    print_loss_total = 0
    loss_history     = []
    iterations       = []

    encoder_optimizer = optim.SGD(encoder.parameters(), lr=learning_rate)
    decoder_optimizer = optim.SGD(decoder.parameters(), lr=learning_rate)
    criterion         = nn.NLLLoss()

    n_iters = len(train_pairs) * epochs
    it = 0

    for epoch in range(epochs):
        print(f"Epoch {epoch + 1}/{epochs}")
        for training_pair in train_pairs:
            input_tensor, target_tensor = tensorsFromPair(training_pair)
            loss = train_with_attention(input_tensor, target_tensor,
                                        encoder, decoder,
                                        encoder_optimizer, decoder_optimizer, criterion)
            print_loss_total += loss
            it += 1
            if it % print_every == 0:
                avg = print_loss_total / print_every
                loss_history.append(avg)
                iterations.append(it)
                print_loss_total = 0
                print(f"{timeSince(start, it / n_iters)} ({it}/{n_iters}) loss={avg:.4f}")

    print("Training complete!")
    return loss_history, iterations


### 5.1 Inference & ROUGE evaluation

In [ ]:
def evaluate(encoder, decoder, sentence, max_length=MAX_LENGTH, encoder_type="rnn"):
    """Greedy decoding for a single sentence."""
    with torch.no_grad():
        input_tensor  = tensorFromSentence(input_lang, sentence)
        input_length  = input_tensor.size(0)
        encoder_hidden = encoder.initHidden()

        if encoder_type == "transformer":
            _, context    = encoder(input_tensor)
            decoder_hidden = context
        else:
            test_output, _ = encoder(input_tensor[0], encoder.initHidden())
            enc_out_size   = test_output.size(2)
            encoder_outputs = torch.zeros(max_length, enc_out_size, device=device)
            encoder_hidden  = encoder.initHidden()

            for ei in range(input_length):
                encoder_output, encoder_hidden = encoder(input_tensor[ei], encoder_hidden)
                encoder_outputs[ei] = encoder_output[0, 0]

            if isinstance(encoder_hidden, tuple):
                h, c = encoder_hidden
                if h.size(0) == 2:
                    h = h.sum(dim=0, keepdim=True)
                    c = c.sum(dim=0, keepdim=True)
                encoder_hidden = (h, c)

            decoder_hidden = encoder_hidden

        decoder_input = torch.tensor([[SOS_token]], device=device)
        decoded_words = []

        for di in range(max_length):
            decoder_output, decoder_hidden = decoder(decoder_input, decoder_hidden)
            topv, topi = decoder_output.data.topk(1)
            if topi.item() == EOS_token:
                decoded_words.append('<EOS>')
                break
            else:
                decoded_words.append(output_lang.index2word[topi.item()])
            decoder_input = topi.squeeze().detach()
            if decoder_input.dim() == 0: decoder_input = decoder_input.view(1, 1)
            elif decoder_input.dim() == 1: decoder_input = decoder_input.view(1, 1)

    return decoded_words


def evaluate_attention(encoder, decoder, sentence, max_length=MAX_LENGTH):
    """Greedy decoding for attention model."""
    with torch.no_grad():
        input_tensor  = tensorFromSentence(input_lang, sentence)
        input_length  = input_tensor.size(0)
        encoder_hidden = encoder.initHidden()
        encoder_outputs = torch.zeros(max_length, encoder.hidden_size, device=device)

        for ei in range(input_length):
            encoder_output, encoder_hidden = encoder(input_tensor[ei], encoder_hidden)
            encoder_outputs[ei] = encoder_output[0, 0]

        decoder_hidden = encoder_hidden
        decoder_input  = torch.tensor([[SOS_token]], device=device)
        decoded_words  = []

        for di in range(max_length):
            decoder_output, decoder_hidden = decoder(decoder_input, decoder_hidden, encoder_outputs)
            topv, topi = decoder_output.data.topk(1)
            if topi.item() == EOS_token:
                decoded_words.append('<EOS>')
                break
            else:
                decoded_words.append(output_lang.index2word[topi.item()])
            decoder_input = topi.squeeze().detach()
            if decoder_input.dim() == 0: decoder_input = decoder_input.view(1, 1)
            elif decoder_input.dim() == 1: decoder_input = decoder_input.view(1, 1)

    return decoded_words


def inference(encoder, decoder, testing_pairs, encoder_type="rnn"):
    input_texts, gt_texts, predict_texts = [], [], []
    for pair in tqdm(testing_pairs):
        output_words   = evaluate(encoder, decoder, pair[0], encoder_type=encoder_type)
        output_sentence = ' '.join(output_words)
        input_texts.append(pair[0])
        gt_texts.append(pair[1])
        predict_texts.append(output_sentence)
    return input_texts, gt_texts, predict_texts


def inference_attention(encoder, decoder, testing_pairs):
    input_texts, gt_texts, predict_texts = [], [], []
    for pair in tqdm(testing_pairs):
        output_words    = evaluate_attention(encoder, decoder, pair[0])
        output_sentence = ' '.join(output_words)
        input_texts.append(pair[0])
        gt_texts.append(pair[1])
        predict_texts.append(output_sentence)
    return input_texts, gt_texts, predict_texts


def eval_rouge(gt, predict, label=""):
    """Compute and print ROUGE-1 and ROUGE-2 F1 scores."""
    rouge        = ROUGEScore()
    metric_score = rouge(predict, gt)
    if label:
        print(f"\n=== ROUGE Scores: {label} ===")
    else:
        print("\n=== ROUGE Scores ===")
    print(f"  ROUGE-1 F1 : {metric_score['rouge1_fmeasure'].item():.4f}")
    print(f"  ROUGE-1 P  : {metric_score['rouge1_precision'].item():.4f}")
    print(f"  ROUGE-1 R  : {metric_score['rouge1_recall'].item():.4f}")
    print(f"  ROUGE-2 F1 : {metric_score['rouge2_fmeasure'].item():.4f}")
    print(f"  ROUGE-2 P  : {metric_score['rouge2_precision'].item():.4f}")
    print(f"  ROUGE-2 R  : {metric_score['rouge2_recall'].item():.4f}")
    return metric_score


## 6. Experiments

### Experiment 1 — GRU Encoder + GRU Decoder (Baseline)

In [ ]:
print("=" * 70)
print("Experiment 1: GRU Encoder + GRU Decoder (Baseline)")
print("=" * 70)

hidden_size = 256
encoder1 = EncoderRNN(input_lang.n_words, hidden_size).to(device)
decoder1 = Decoder(hidden_size, output_lang.n_words).to(device)

loss_hist1, iters1 = trainIters(encoder1, decoder1, epochs=3,
                                print_every=2000, encoder_type="rnn")

input1, gt1, pred1 = inference(encoder1, decoder1, test_pairs, encoder_type="rnn")
rouge1 = eval_rouge(gt1, pred1, label="GRU Baseline")


### Experiment 2 — LSTM Encoder + LSTM Decoder

In [ ]:
print("=" * 70)
print("Experiment 2: LSTM Encoder + LSTM Decoder")
print("=" * 70)

encoder2 = EncoderLSTM(input_lang.n_words, hidden_size).to(device)
decoder2 = DecoderLSTM(hidden_size, output_lang.n_words).to(device)

loss_hist2, iters2 = trainIters(encoder2, decoder2, epochs=3,
                                print_every=2000, encoder_type="rnn")

input2, gt2, pred2 = inference(encoder2, decoder2, test_pairs, encoder_type="rnn")
rouge2 = eval_rouge(gt2, pred2, label="LSTM")


### Experiment 3 — Bi-LSTM Encoder + LSTM Decoder

In [ ]:
print("=" * 70)
print("Experiment 3: Bi-LSTM Encoder + LSTM Decoder")
print("=" * 70)

encoder3 = EncoderBiLSTM(input_lang.n_words, hidden_size).to(device)
decoder3 = DecoderLSTM(hidden_size, output_lang.n_words).to(device)

loss_hist3, iters3 = trainIters(encoder3, decoder3, epochs=3,
                                print_every=2000, encoder_type="rnn")

input3, gt3, pred3 = inference(encoder3, decoder3, test_pairs, encoder_type="rnn")
rouge3 = eval_rouge(gt3, pred3, label="Bi-LSTM")


### Experiment 4 — GRU Encoder + GRU Decoder with Attention (Best Model)

In [ ]:
print("=" * 70)
print("Experiment 4: GRU Encoder + GRU Decoder with Attention")
print("=" * 70)

encoder4 = EncoderRNN(input_lang.n_words, hidden_size).to(device)
decoder4 = DecoderWithAttention(hidden_size, output_lang.n_words).to(device)

loss_hist4, iters4 = trainIters_attention(encoder4, decoder4, epochs=3,
                                          print_every=2000)

input4, gt4, pred4 = inference_attention(encoder4, decoder4, test_pairs)
rouge4 = eval_rouge(gt4, pred4, label="GRU + Attention")


### Experiment 5 — Transformer Encoder + GRU Decoder

In [ ]:
print("=" * 70)
print("Experiment 5: Transformer Encoder + GRU Decoder")
print("=" * 70)

encoder5 = TransformerEncoder(input_lang.n_words, hidden_size, num_layers=2).to(device)
decoder5 = Decoder(hidden_size, output_lang.n_words).to(device)

loss_hist5, iters5 = trainIters(encoder5, decoder5, epochs=3,
                                print_every=2000, encoder_type="transformer")

input5, gt5, pred5 = inference(encoder5, decoder5, test_pairs, encoder_type="transformer")
rouge5 = eval_rouge(gt5, pred5, label="Transformer")


## 7. Results & Visualisation

### 7.1 ROUGE scores summary table

In [ ]:
results_df = pd.DataFrame({
    'Model': ['GRU (Baseline)', 'LSTM', 'Bi-LSTM', 'GRU + Attention', 'Transformer'],
    'ROUGE-1 F1': [
        rouge1['rouge1_fmeasure'].item(),
        rouge2['rouge1_fmeasure'].item(),
        rouge3['rouge1_fmeasure'].item(),
        rouge4['rouge1_fmeasure'].item(),
        rouge5['rouge1_fmeasure'].item(),
    ],
    'ROUGE-2 F1': [
        rouge1['rouge2_fmeasure'].item(),
        rouge2['rouge2_fmeasure'].item(),
        rouge3['rouge2_fmeasure'].item(),
        rouge4['rouge2_fmeasure'].item(),
        rouge5['rouge2_fmeasure'].item(),
    ],
})
results_df['Δ ROUGE-1 vs Baseline'] = results_df['ROUGE-1 F1'] - results_df.loc[0, 'ROUGE-1 F1']
results_df['Δ ROUGE-2 vs Baseline'] = results_df['ROUGE-2 F1'] - results_df.loc[0, 'ROUGE-2 F1']

print("\n" + "=" * 70)
print("FINAL RESULTS SUMMARY")
print("=" * 70)
print(results_df.to_string(index=False))
print("\n✅ All experiments completed successfully!")


### 7.2 ROUGE bar chart

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52', '#8172B3']
models = results_df['Model'].tolist()
x      = range(len(models))

for ax, metric, title in zip(axes,
                              ['ROUGE-1 F1', 'ROUGE-2 F1'],
                              ['ROUGE-1 F1 Score', 'ROUGE-2 F1 Score']):
    bars = ax.bar(x, results_df[metric], color=colors, edgecolor='white', linewidth=0.8)
    ax.set_xticks(list(x))
    ax.set_xticklabels(models, rotation=20, ha='right', fontsize=10)
    ax.set_ylabel('F1 Score', fontsize=11)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_ylim(0, 0.75)
    ax.axhline(results_df.loc[0, metric], color='grey', linestyle='--',
               linewidth=1, alpha=0.6, label='Baseline')
    ax.legend(fontsize=9)
    for bar, val in zip(bars, results_df[metric]):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                f'{val:.4f}', ha='center', va='bottom', fontsize=8.5)

plt.suptitle('Architecture Comparison — English→French Translation',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('rouge_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Chart saved as rouge_comparison.png")


### 7.3 Training loss curves

In [ ]:
# Populate loss histories from recorded printed output
# (replace with live loss_hist* variables when retraining from scratch)
gru_losses   = [3.4534,3.0385,2.8872,2.7040,2.6234,2.5161,2.3848,2.3277,2.2876,
                2.2192,2.1139,2.0483,1.9326,1.9189,1.8882,1.8177,1.7907,1.7507,1.7498,
                1.6997,1.5997,1.5621,1.5311,1.5331,1.4826,1.4596,1.4255,1.4079]
attn_losses  = [3.6564,3.0926,2.7879,2.5958,2.4830,2.3369,2.2773,2.1851,2.1425,
                2.0568,1.9455,1.8886,1.7355,1.7665,1.7373,1.6672,1.6589,1.6147,1.6113,
                1.5698,1.4685,1.4218,1.4279,1.4225,1.3818,1.3923,1.3210,1.3266]
lstm_losses  = [3.5460,3.0857,2.9878,2.8654,2.8102,2.7146,2.6160,2.5548,2.5176,
                2.4709,2.3231,2.2836,2.1901,2.1624,2.1658,2.0746,2.0354,2.0136,2.0072,
                1.9349,1.8296,1.8112,1.7874,1.7606,1.7461,1.6867,1.6639,1.6486]
bilstm_losses= [3.5200,3.1482,2.9798,2.8289,2.7610,2.6881,2.5781,2.5269,2.4928,
                2.4385,2.3030,2.2398,2.1483,2.1355,2.1172,2.0518,1.9998,1.9713,1.9752,
                1.9081,1.7938,1.7708,1.7454,1.7439,1.7116,1.6794,1.6470,1.6301]
trans_losses = [3.2767,2.9057,2.7552,2.6363,2.5765,2.5029,2.4154,2.4069,2.3829,
                2.3323,2.2292,2.1929,2.1174,2.1319,2.1005,2.0783,2.0573,2.0216,2.0568,
                1.9999,1.8903,1.8759,1.8754,1.8603,1.8323,1.8407,1.8154,1.8012]
iters        = list(range(2000, 57316, 2000))

fig, ax = plt.subplots(figsize=(12, 6))
for losses, label, marker in [
    (gru_losses,    'GRU Baseline',     'o'),
    (attn_losses,   'GRU + Attention',  's'),
    (lstm_losses,   'LSTM',             '^'),
    (bilstm_losses, 'Bi-LSTM',          'v'),
    (trans_losses,  'Transformer',      'd'),
]:
    ax.plot(iters[:len(losses)], losses, label=label, marker=marker,
            linewidth=2, markersize=4, alpha=0.85)

for em in [19105, 38210]:
    ax.axvline(x=em, color='grey', linestyle='--', alpha=0.4, linewidth=1)

ax.set_xlabel('Training Iterations', fontsize=12)
ax.set_ylabel('Average NLL Loss',    fontsize=12)
ax.set_title('Training Loss Curves — All Architectures (3 Epochs)',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10, loc='upper right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('loss_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print("Loss curves saved as loss_curves.png")


## 8. Analysis & Discussion

### 8.1 Best architecture — GRU + Attention

The attention-augmented GRU achieved the highest ROUGE-1 (0.6309) and ROUGE-2 (0.4360), 
outperforming the baseline by +2.7% and +1.9% respectively, with the lowest final training 
loss (1.33). The vectorised multiplicative attention allows the decoder to selectively focus 
on relevant encoder states at each decoding step, improving alignment between source and target.

### 8.2 LSTM / Bi-LSTM underperformance

Both LSTM variants underperformed the simpler GRU baseline on this short, heavily filtered 
dataset (avg ~6 words/sentence). Three plausible explanations:

1. **Dataset simplicity** — GRU's two-gate mechanism may generalise better than LSTM's three 
   gates on very short sequences, where additional expressiveness becomes overfitting risk.
2. **Crude Bi-LSTM merging** — Summing forward/backward hidden states discards directional 
   information. Concatenation + linear projection would be preferable.
3. **Optimisation difficulty** — More parameters make the loss landscape harder to navigate 
   with vanilla SGD; momentum or adaptive rates would likely help.

### 8.3 Transformer encoder limitations

The Transformer achieved the worst ROUGE scores (−13% / −18.3% vs baseline) despite the 
fastest wall-clock training. Root causes:

- **Learnable positional embeddings** do not generalise well to short sequences without regularisation.
- **Mean pooling** discards position-specific information critical for translation.
- **No pre-training** — 19K pairs are insufficient for self-attention to learn useful patterns.

### 8.4 Future improvements

| Direction | Recommendation |
|-----------|---------------|
| Bi-LSTM merging | Replace sum with `Linear(cat([h_fwd, h_bwd]))` |
| Transformer PE | Use sinusoidal instead of learnable embeddings |
| Transformer pooling | Replace mean with attention-based or CLS-token pooling |
| Data scale | Augment to 50K+ pairs via back-translation |
| Optimiser | Warmup + cosine-annealing LR schedule for LSTM / Transformer |
